In [ ]:
import os
import xarray as xr
import pandas as pd

# Define the location of interest
target_lat = 37.0443931
target_lon = -122.072464

# Define input and output paths
input_dir = "/data/muscat_data/jaguir26/project1_ucsc_phd/soil_moisture_data"
output_csv = os.path.join(input_dir, "soil_moisture_big_trees_hourly_1987_2023.csv")

# List only valid unzipped .nc files
nc_files = sorted([
    os.path.join(input_dir, f)
    for f in os.listdir(input_dir)
    if f.endswith(".nc") and "unzipped" in f
])

all_data = []

# Loop through files and extract time series
for f in nc_files:
    print(f"📂 Processing: {f}")
    try:
        ds = xr.open_dataset(f, engine="netcdf4")
        point_data = ds.sel(latitude=target_lat, longitude=target_lon, method="nearest")
        df = point_data["swvl1"].to_dataframe().reset_index()
        df = df[['valid_time', 'swvl1']].rename(columns={
            'valid_time': 'Date',
            'swvl1': 'Soil_Moisture'
        })
        all_data.append(df)
        ds.close()
    except Exception as e:
        print(f"❌ Error with file {f}: {e}")
        continue

# Combine all data and save
if all_data:
    combined_df = pd.concat(all_data)
    combined_df = combined_df.sort_values("Date").drop_duplicates(subset="Date")
    combined_df.to_csv(output_csv, index=False)
    print(f"\n✅ Combined soil moisture time series saved to:\n{output_csv}")
else:
    print("⚠️ No data extracted.")


In [ ]:
import pandas as pd

# Load your hourly CSV
csv_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/soil_moisture_data/soil_moisture_big_trees_hourly_1987_2023.csv"
df = pd.read_csv(csv_path, parse_dates=["Date"])

# Compute daily average by grouping on just the date part
df['Day'] = df['Date'].dt.date
daily_avg = df.groupby('Day')['Soil_Moisture'].mean().reset_index()

# Rename for clarity
daily_avg = daily_avg.rename(columns={'Day': 'Date', 'Soil_Moisture': 'Daily_Avg_Soil_Moisture'})

# Save to CSV
output_path = csv_path.replace("hourly", "daily_avg")
daily_avg.to_csv(output_path, index=False)

print(f"✅ Daily average soil moisture saved to:\n{output_path}")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the daily average soil moisture data
csv_path = "/data/muscat_data/jaguir26/project1_ucsc_phd/soil_moisture_data/soil_moisture_big_trees_daily_avg_1987_2023.csv"
df = pd.read_csv(csv_path, parse_dates=["Date"])

# Create the plot
plt.figure(figsize=(14, 6))
plt.plot(df["Date"], df["Daily_Avg_Soil_Moisture"], color="forestgreen", linewidth=0.8)
plt.title("Daily Average Soil Moisture at San Lorenzo River (1987–2023)", fontsize=14)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Soil Moisture (m³/m³)", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()

#
